In [2]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy import stats
from pathlib import Path
import sys
import os
from datetime import datetime
from dataScraper import *

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import nameDict


pd.set_option('display.max_columns', None)

In [3]:
today = datetime.today().strftime('%Y%m%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

us_file = get_latest_file('NBA_US_*.csv')
dfs_file = get_latest_file('NBA_DFS_*.csv')

if us_file is None:
    raise ValueError("No US file found")

if dfs_file is None:
    raise ValueError("No DFS file found")

us_df = pd.read_csv(us_file)
lines_dfs = pd.read_csv(dfs_file)

lines_us = us_df[us_df['CATEGORY'] == 'player_points'].copy()


print("US file:", us_file.name)
print("DFS file:", dfs_file.name)
print("DFS latest pull:", lines_dfs['DATA_PULLED_AT'].max())
print("US latest pull:", us_df['DATA_PULLED_AT'].max())

US file: NBA_US_20260321_150010.csv
DFS file: NBA_DFS_20260321_145905.csv
DFS latest pull: 2026-03-21 14:59:05
US latest pull: 2026-03-21 15:00:10


In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')

if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260321_150009.json


,home_team,away_team,commence_time,bookmakers
0,Washington Wizards,Oklahoma City Thunder,2026-03-21 21:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
1,Charlotte Hornets,Memphis Grizzlies,2026-03-21 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
2,New Orleans Pelicans,Cleveland Cavaliers,2026-03-21 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
3,Orlando Magic,Los Angeles Lakers,2026-03-21 23:11:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
4,Atlanta Hawks,Golden State Warriors,2026-03-22 00:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [5]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s25, s26])
base_df.tail()

,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE
86,2025-26,203991,Clint Capela,Clint,1610612745,HOU,Houston Rockets,22501018,2026-03-20T00:00:00,HOU vs. ATL,W,13.791667,1,2,0.500,0,0,0.0,1,4,0.25,2,3,5,2,1,1,0,1,0,2,3,8,14.0,0,0,12.0,1,13:48,1,111.5,112.9,112.9,89.8,87.1,87.1,21.7,25.8,25.8,0.182,2.0,28.6,0.125,0.214,0.167,14.3,14.8,0.500,0.399,0.135,0.127,106.99,107.89,89.91,107.89,0.084,31,1.0,2.0,42,83,0.506,14,30,0.467,19,28,0.679,12,39,51,33,20.0,11,5,9,13,21,117,22.0,113.2,117.0,93.6,93.1,19.6,23.9,0.786,1.65,22.1,0.348,0.755,0.566,0.200,0.590,0.614,102.4,101.0,84.17,100,0.623,1610612737,ATL,Atlanta Hawks,36,85,0.424,9,35,0.257,14,17,0.824,9,28,37,22,18.0,13,9,5,21,13,95,-22.0,93.6,93.1,113.2,117.0,-19.6,-23.9,0.611,1.22,16.4,0.245,0.652,0.434,0.176,0.476,0.514,102.4,101.0,84.17,102,0.377
87,2025-26,1642864,Hugo González,Hugo,1610612738,BOS,Boston Celtics,22501019,2026-03-20T00:00:00,BOS @ MEM,W,14.561667,2,2,1.000,1,1,1.0,0,0,0.00,0,5,5,1,0,0,0,0,3,0,5,7,12.5,0,0,12.0,1,14:34,1,142.4,143.3,143.3,121.5,120.0,120.0,20.9,23.3,23.3,0.067,0.0,33.3,0.000,0.333,0.208,0.0,0.0,1.250,1.250,0.061,0.062,98.63,98.89,82.41,98.89,0.110,30,2.0,2.0,40,89,0.449,11,42,0.262,26,30,0.867,18,39,57,19,13.0,4,2,3,20,23,117,5.0,120.4,120.6,112.6,115.5,7.8,5.2,0.475,1.46,13.9,0.404,0.784,0.592,0.134,0.511,0.572,98.3,97.0,80.83,97,0.526,1610612763,MEM,Memphis Grizzlies,42,90,0.467,14,43,0.326,14,17,0.824,7,28,35,24,9.0,7,3,2,23,20,112,-5.0,112.6,115.5,120.4,120.6,-7.8,-5.2,0.571,2.67,18.5,0.216,0.596,0.408,0.093,0.544,0.574,98.3,97.0,80.83,97,0.474
88,2025-26,1631248,Baylor Scheierman,Baylor,1610612738,BOS,Boston Celtics,22501019,2026-03-20T00:00:00,BOS @ MEM,W,23.216667,1,2,0.500,0,1,0.0,0,0,0.00,2,2,4,3,0,1,0,0,1,2,2,7,14.3,0,0,11.0,1,23:13,1,132.2,130.6,130.6,115.4,116.3,116.3,16.8,14.3,14.3,0.125,0.0,60.0,0.091,0.080,0.085,0.0,0.0,0.500,0.500,0.035,0.036,101.10,101.31,84.42,101.31,0.067,49,1.0,2.0,40,89,0.449,11,42,0.262,26,30,0.867,18,39,57,19,13.0,4,2,3,20,23,117,5.0,120.4,120.6,112.6,115.5,7.8,5.2,0.475,1.46,13.9,0.404,0.784,0.592,0.134,0.511,0.572,98.3,97.0,80.83,97,0.526,1610612763,MEM,Memphis Grizzlies,42,90,0.467,14,43,0.326,14,17,0.824,7,28,35,24,9.0,7,3,2,23,20,112,-5.0,112.6,115.5,120.4,120.6,-7.8,-5.2,0.571,2.67,18.5,0.216,0.596,0.408,0.093,0.544,0.574,98.3,97.0,80.83,97,0.474
90,2025-26,1630700,Dyson Daniels,Dyson,1610612737,ATL,Atlanta Hawks,22501018,2026-03-20T00:00:00,ATL @ HOU,L,25.166667,1,5,0.200,0,0,0.0,1,2,0.50,0,3,3,

In [6]:
# Change line_bookmaker to e.g. 'PrizePicks' to use that DFS book's lines from lines_dfs
final, tier1_all, final = generalized_best_bets(
    lines_dfs, base_df, us_df, team_dds, nameDict,
    line_bookmaker='Underdog',
)

print('Total bets across categories:', len(final))
print('Tier 1 bets:', len(tier1_all))

if not final.empty:
    display(tier1_all.head(20))

/Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/points_model/dataScraper.py:218: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cover = cover_df.groupby('PLAYER_NAME').apply(
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/points_model/dataScraper.py:218: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cover = cover_df.groupby('PLAYER_NAME').apply(
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/src/points_mo

Total bets across categories: 247
Tier 1 bets: 42


,PLAYER_NAME,TEAM_NAME,OPPONENT,HOME_AWAY,TEAM_SPREAD,GAME_TOTAL,CATEGORY,LINE,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES,MATCHUP_EDGE,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,TOTAL_BOOST,IS_UNDERDOG,BET_FLAG,COMMENCE_TIME
0,Daniel Gafford,Dallas Mavericks,Los Angeles Clippers,HOME,7.5,233.5,player_points,9.5,-106,-112,0.515,0.528,14.2,13.0,7.25,4.0,-2.25,6.05,4.7,3.5,-0.777,0.781,0.219,51.78,-58.55,1.0,0.7,0.53,0.56,22.53,2.98,0.19,0.03,1.35,1,True,2026-03-22
1,Amen Thompson,Houston Rockets,Miami Heat,HOME,-2.0,227.5,player_points,18.5,105,-116,0.488,0.537,21.0,22.5,14.33,3.0,-4.17,4.08,2.5,4.0,-0.613,0.730,0.270,49.65,-49.72,0.6,0.7,0.60,0.37,37.53,4.60,0.20,0.03,0.75,0,True,2026-03-22
2,Jay Huff,Indiana Pacers,San Antonio Spurs,AWAY,17.5,235.0,player_points,9.5,-118,105,0.541,0.488,12.8,13.0,3.40,5.0,-6.10,3.82,3.3,3.5,-0.864,0.806,0.194,48.91,-60.23,0.6,0.8,0.67,0.42,23.16,6.38,0.23,0.05,1.50,1,True,2026-03-22
3,Bam Adebayo,Miami Heat,Houston Rockets,AWAY,2.0,227.5,player_points,21.5,105,-115,0.488,0.535,29.7,24.0,17.33,3.0,-4.17,18.95,8.2,2.5,-0.433,0.667,0.333,36.73,-37.74,0.6,0.7,0.60,0.34,35.70,4.46,0.31,0.09,0.75,1,True,2026-03-22
4,Rui Hachimura,Los Angeles Lakers,Orlando Magic,AWAY,-3.5,233.5,player_points,7.5,100,-110,0.500,0.524,9.5,8.5,7.50,2.0,0.00,4.55,2.0,1.0,-0.440,0.670,0.330,34.00,-37.00,0.4,0.6,0.60,0.79,25.88,7.46,0.13,0.03,1.35,0,True,2026-03-21
5,Pelle Larsson,Miami Heat,Houston Rockets,AWAY,2.0,227.5,player_points,11.5,110,-104,0.476,0.510,13.8,12.5,20.00,1.0,8.50,6.73,2.3,1.0,-0.342,0.634,0.366,33.14,-28.21,0.6,0.5,0.40,0.27,30.32,5.75,0.18,0.04,0.75,1,True,2026-03-22
6,Jevon Carter,Orlando Magic,Los Angeles Lakers,HOME,3.5,233.5,player_points,7.5,-105,105,0.512,0.488,8.6,8.5,0.67,3.0,-6.83,2.63,1.1,1.0,-0.418,0.662,0.338,29.25,-30.71,0.6,0.6,0.53,0.32,20.63,3.96,0.19,0.05,1.35,1,True,2026-03-21
7,Marcus Smart,Los Angeles Lakers,Orlando Magic,AWAY,-3.5,233.5,player_points,8.5,-108,-103,0.519,0.507,10.2,9.5,8.00,3.0,-0.50,4.57,1.7,1.0,-0.372,0.645,0.355,24.22,-30.03,0.6,0.7,0.53,0.56,30.88,3.52,0.14,0.04,1.35,0,True,2026-03-21
8,Klay Thompson,Dallas Mavericks,Los Angeles Clippers,HOME,7.5,233.5,player_points,10.5,-116,-104,0.537,0.510,13.3,13.0,15.00,6.0,4.50,7.01,2.8,2.5,-0.399,0.655,0.345,21.97,-32.33,0.6,0.5,0.47,0.61,20.33,4.38,0.25,0.04,1.35,1,True,2026-03-22
9,Taylor Hendricks,Memphis Grizzlies,Charlotte Hornets,AWAY,18.5,233.0,player_points,11.5,-111,-114,0.526,0.533,12.9,12.5,10.50,2.0,-1.00,4.43,1.4,1.0,-0.316,0.624,0.376,18.62,-29.42,0.8,0.6,0.53,0.26,25.04,2.48,0.18,0.06,1.30,1,True,2026-03-21


In [7]:
rename_map = {
    'PLAYER_NAME': 'Player',
    'CATEGORY': 'Prop',
    'LINE': 'Line',
    'OPPONENT': 'Opponent',
    'TEAM_SPREAD': 'Spread',
    'GAME_TOTAL': 'Total',
    'OPP_DEF_RATING': 'Opp Def Rating',
    'OPP_RANK_DEF_RATING': 'Opp Def Rank',
    'OPP_PACE': 'Opp Pace',
    'OPP_PACE_RANK': 'Opp Pace Rank',
    'ODDS_OVER': 'Odds Over',
    'ODDS_UNDER': 'Odds Under',
    'IMP_PROB_OVER': 'Implied Over',
    'IMP_PROB_UNDER': 'Implied Under',
    'AVG_STAT_L10': 'Avg Stat L10',
    'MED_STAT_L10': 'Med Stat L10',
    'STD_STAT_L10': 'Std Stat L10',
    'EDGE': 'Edge',
    'MED_EDGE': 'Med Edge',
    'Z_SCORE': 'Z Score',
    'PROB_OVER': 'Prob Over',
    'PROB_UNDER': 'Prob Under',
    'EV_OVER': 'EV Over',
    'EV_UNDER': 'EV Under',
    'OVER_RATE_L5': 'OVER L5',
    'OVER_RATE_L10': 'OVER L10',
    'OVER_RATE_L15': 'OVER L15',
    'OVER_RATE_SEASON': 'ALL SEASON',
    'AVG_MIN_L10': 'Avg Min L10',
    'STD_MIN_L10': 'Std Min L10',
    'AVG_USG_L10': 'Avg USG% L10',
    'STD_USG_L10': 'Std USG% L10',
    'MIN_CONSISTENCY': 'Min Consistency',
    'IS_UNDERDOG': 'Underdog',
    'AVG_STAT_VS_MATCHUP': 'Avg Stat vs Matchup',
    'MATCHUP_GAMES': 'Matchup Games',
}

df = final.rename(columns=rename_map)
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['Prop'] = df['Prop'].map(prop_label_map).fillna(df['Prop'])

df = df[[
    'Player',
    'Prop',
    'Line',
    'Opponent',
    'Odds Over',
    'Odds Under',
    'Implied Over',
    'Implied Under',
    'EV Over',
    'EV Under',
    'Avg Stat L10',
    'Med Stat L10',
    'Std Stat L10',
    'Z Score',
    'Prob Over',
    'Prob Under',
    'OVER L5',
    'OVER L10',
    'OVER L15',
    'Avg Min L10',
    'Std Min L10',
    'Avg USG% L10',
    'Std USG% L10',
    'Avg Stat vs Matchup',
    'Matchup Games',
    'Spread',
    'Total',
    'Opp Def Rating',
    'Opp Def Rank',
    'Opp Pace',
    'Opp Pace Rank',
]].sort_values(by='EV Over', ascending=False)
df.head(10)

,Player,Prop,Line,Opponent,Odds Over,Odds Under,Implied Over,Implied Under,EV Over,EV Under,Avg Stat L10,Med Stat L10,Std Stat L10,Z Score,Prob Over,Prob Under,OVER L5,OVER L10,OVER L15,Avg Min L10,Std Min L10,Avg USG% L10,Std USG% L10,Avg Stat vs Matchup,Matchup Games,Spread,Total,Opp Def Rating,Opp Def Rank,Opp Pace,Opp Pace Rank
110,Pascal Siakam,PTS+REB+AST,27.5,San Antonio Spurs,110,-115,0.476,0.535,80.60,-73.83,33.2,33.5,5.27,-1.082,0.860,0.140,1.0,0.9,0.93,31.70,3.69,0.31,0.04,34.00,3.0,17.5,235.0,110.5,3.0,100.77,12.0
0,Daniel Gafford,PTS,9.5,Los Angeles Clippers,-106,-112,0.515,0.528,51.78,-58.55,14.2,13.0,6.05,-0.777,0.781,0.219,1.0,0.7,0.53,22.53,2.98,0.19,0.03,7.25,4.0,7.5,233.5,115.7,19.0,97.15,28.0
61,Keon Ellis,REB,2.5,New Orleans Pelicans,-114,-104,0.533,0.510,50.18,-60.77,3.3,3.0,0.95,-0.842,0.800,0.200,0.8,0.8,0.73,27.62,2.91,0.12,0.06,4.00,4.0,-5.5,239.0,117.0,24.0,101.10,11.0
1,Amen Thompson,PTS,18.5,Miami Heat,105,-116,0.488,0.537,49.65,-49.72,21.0,22.5,4.08,-0.613,0.730,0.270,0.6,0.7,0.60,37.53,4.60,0.20,0.03,14.33,3.0,-2.0,227.5,112.1,6.0,104.52,1.0
2,Jay Huff,PTS,9.5,San Antonio Spurs,-118,105,0.541,0.488,48.91,-60.23,12.8,13.0,3.82,-0.864,0.806,0.194,0.6,0.8,0.67,23.16,6.38,0.23,0.05,3.40,5.0,17.5,235.0,110.5,3.0,100.77,12.0
98,Marcus Smart,3PM,1.5,Orlando Magic,104,-130,0.490,0.565,39.13,-43.74,2.2,2.5,1.48,-0.473,0.682,0.318,0.4,0.6,0.53,30.88,3.52,0.14,0.04,1.00,3.0,-3.5,233.5,113.2,12.0,100.12,18.0
196,Bam Adebayo,PTS+AST,24.5,Houston Rockets,110,-105,0.476,0.512,39.02,-34.01,32.5,27.5,19.13,-0.418,0.662,0.338,0.6,0.7,0.60,35.70,4.46,0.31,0.09,20.00,3.0,2.0,227.5,112.3,8.0,96.71,29.0
230,Austin Reaves,REB+AST,9.5,Orlando Magic,102,-115,0.495,0.535,38.98,-41.67,10.7,11.0,2.45,-0.490,0.688,0.312,0.6,0.6,0.40,38.34,4.16,0.23,0.05,6.67,3.0,-3.5,233.5,113.2,12.0,100.12,18.0
168,Amen Thompson,PTS+REB,26.5,Miami Heat,-105,-111,0.512,0.526,38.81,-45.06,29.6,30.5,5.58,-0.556,0.711,0.289,0.6,0.7,0.60,37.53,4.60,0.20,0.03,22.67,3.0,-2.0,227.5,112.1,6.0,104.52,1.0
3,Bam Adebayo,PTS,21.5,Houston Rockets,105,-115,0.488,0.535,36.73,-37.74,29.7,24.0,18.95,-0.433,0.667,0.333,0.6,0.7,0.60,35.70,4.46,0.31,0.09,17.33,3.0,2.0,227.5,112.3,8.0,96.71,29.0


In [8]:
output_path = f'data/props/ev_analysis/underdog.csv'
# output_path = f'data/props/ev_analysis/prizepicks.csv'
df.to_csv(output_path, index=False)